In [1]:
import sqlalchemy
from sqlalchemy import create_engine, text
import cx_Oracle
from IPython.core.magic import register_cell_magic
import sys
import pandas as pd

if sys.version.find('GCC') >= 0:
    domain_name = 'oracle-db'
else:
    domain_name = 'localhost'
    cx_Oracle.init_oracle_client(lib_dir = r"C:\Oracle\instantclient_23_7")
DATABASE_URL = "oracle+cx_oracle://lib:multisqld@{}:1521/?service_name=XEPDB1".format(domain_name)
engine = create_engine(DATABASE_URL)
def get_rows(qry):
    try:
        with engine.connect() as conn:
            result = conn.execute(text(qry))
            df = pd.DataFrame(result.fetchall(), columns = result.keys()).fillna('NULL').pipe(lambda x: x.set_index(x.index + 1))
            return df if len(df) != 1 else df.iloc[0]
    except sqlalchemy.exc.DatabaseError as e:
        orig = e.orig
        offset = getattr(orig, 'offset', None)
        message = getattr(orig, 'message', str(orig))
        print(message, offset)
    except Exception as e:
        print(e.__class__)
        print(e)

def exec_qrys(qrys):
    try:
        results = list()
        with engine.connect() as conn:
            for qry in qrys.split(';'):
                qry = qry.strip()
                if len(qry) == 0:
                    continue
                result = conn.execute(text(qry))
                results.append(
                    "{}, 변경된 행의 수: {}".format(qry, result.rowcount)
                )
        return results
    except Exception as e:
        print(e)

def exec_qry(qry):
    try:
        with engine.connect() as conn:
            result = conn.execute(text(qry))
            conn.commit()
            return "변경된 행의 수: {}".format(result.rowcount)
    except Exception as e:
        print(e)

@register_cell_magic
def SQL(line, cell):
    spt = [i for i in cell.split(';') if len(i.strip()) > 0]
    if len(spt) > 1:
        return exec_qrys(cell)
    sql = cell.strip().replace(';', '')
    if sql.lower().startswith('select'):
        return get_rows(sql)
    else:
        return exec_qry(sql)

# DML(Data Manipulation Language)

|명령|용법|
|----|----|
|INSERT|INSERT INTO 테이블명 [(컬럼1, 컬럼2, … )] <br/> {VALUES (값1, 값2, …)| 서브쿼리};|
|UPDATE|UPDATE 테이블 명 SET<br/>수정할 컬럼1 = 표현1<br/>[, 수정할 컬럼2 = 표현2] …<br/>[WHERE 절]|
|DELETE|DELETE [FROM] 테이블명 [WHERE 절]|
|MERGE|MERGE INTO 타겟 테이블명<br/>USING 소스 테이블명 ON (일치 판별식)<br/>WHEN MATCHED THEN<br/>UPDATE SET<br/>수정할 컬럼1 = 표현1<br/>[, 수정할 컬럼2 = 표현2] …<br/>WHEN NOT MATCHED THEN<br/>INSERT [(컬럼1, 컬럼2, …)]<br/>VALUES (값1, 값2, …)}|

**[예제1]** DML 연습을 해봅니다.

- 연습용 테이블 추가 명령 입니다.

In [2]:
exec_qry("""
CREATE TABLE book_info (
    book_id NUMBER(12),
    title VARCHAR2(200),
    publisher VARCHAR2(100),
    reg_date DATE,
    CONSTRAINT PK_book_info PRIMARY KEY (book_id)
)"""
)

(cx_Oracle.DatabaseError) ORA-00955: name is already used by an existing object
[SQL: 
CREATE TABLE book_info (
    book_id NUMBER(12),
    title VARCHAR2(200),
    publisher VARCHAR2(100),
    reg_date DATE,
    CONSTRAINT PK_book_info PRIMARY KEY (book_id)
)]
(Background on this error at: https://sqlalche.me/e/20/4xp6)


In [3]:
%%SQL

SELECT * FROM book_info

,book_id,title,publisher,reg_date
1,1,NULL,영국출판사~~~,2025-05-06 00:00:00
2,2459,수상한 과학,풀빛,2026-05-20 09:05:30
3,14,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2026-05-20 09:05:30
4,1384,2024 SIMPLE 의료법규,군자출판사,2026-05-20 09:05:30
5,704,임신부·모유수유부의 안전한 약물 사용,군자출판사,2026-05-20 09:05:30
6,1706,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2026-05-20 09:05:30
7,1065,상실의 시대,문학사상,2026-05-20 09:05:30
8,1544,"궁금해요, 세종 대왕",풀빛,2026-05-20 09:05:30
9,971,뉴스를 보는 눈,풀빛,2026-05-20 09:05:30
10,1873,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2026-05-20 09:05:30


- INSERT를 통해 레코드를 하나 추가합니다.

  INSERT INTO 테이블명(컬럼명|[컬럼명2], ...) VALUES (....)

In [4]:
%%SQL

INSERT INTO book_info(book_id, title, publisher, reg_date)
VALUES (1, '해리포터', '외국출판사', to_date('2025-05-07', 'yyyy-mm-dd'))

(cx_Oracle.IntegrityError) ORA-00001: unique constraint (LIB.PK_BOOK_INFO) violated
[SQL: INSERT INTO book_info(book_id, title, publisher, reg_date)
VALUES (1, '해리포터', '외국출판사', to_date('2025-05-07', 'yyyy-mm-dd'))]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [5]:
%%SQL

SELECT * FROM book_info

,book_id,title,publisher,reg_date
1,1,NULL,영국출판사~~~,2025-05-06 00:00:00
2,2459,수상한 과학,풀빛,2026-05-20 09:05:30
3,14,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2026-05-20 09:05:30
4,1384,2024 SIMPLE 의료법규,군자출판사,2026-05-20 09:05:30
5,704,임신부·모유수유부의 안전한 약물 사용,군자출판사,2026-05-20 09:05:30
6,1706,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2026-05-20 09:05:30
7,1065,상실의 시대,문학사상,2026-05-20 09:05:30
8,1544,"궁금해요, 세종 대왕",풀빛,2026-05-20 09:05:30
9,971,뉴스를 보는 눈,풀빛,2026-05-20 09:05:30
10,1873,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2026-05-20 09:05:30


- 하나 더 추가합니다.

In [6]:
%%SQL

INSERT INTO book_info
VALUES (1, '반지의 제왕', '보석출판사', to_date('2025_06-01', 'yyyy-mm-dd'))

(cx_Oracle.IntegrityError) ORA-00001: unique constraint (LIB.PK_BOOK_INFO) violated
[SQL: INSERT INTO book_info
VALUES (1, '반지의 제왕', '보석출판사', to_date('2025_06-01', 'yyyy-mm-dd'))]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


인덱스가 중복하기 때문에 추가가 되지 않았습니다.

- 컬럼을 지정하지 않으면 등록된 컬럼 순에 맞춰 입력이 됩니다. 이렇게 쓰면 부작용이 있어 쓰지 않습니다.  개념과 외부 스키마 간에 종속성이 생깁니다.

In [7]:
%%SQL

INSERT INTO book_info
VALUES (2, '반지의 제왕', '보석출판사', to_date('2025_06-01', 'yyyy-mm-dd'))

'변경된 행의 수: 1'

In [8]:
%%SQL

SELECT * FROM book_info

,book_id,title,publisher,reg_date
1,1,NULL,영국출판사~~~,2025-05-06 00:00:00
2,2459,수상한 과학,풀빛,2026-05-20 09:05:30
3,14,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2026-05-20 09:05:30
4,1384,2024 SIMPLE 의료법규,군자출판사,2026-05-20 09:05:30
5,704,임신부·모유수유부의 안전한 약물 사용,군자출판사,2026-05-20 09:05:30
6,1706,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2026-05-20 09:05:30
7,1065,상실의 시대,문학사상,2026-05-20 09:05:30
8,1544,"궁금해요, 세종 대왕",풀빛,2026-05-20 09:05:30
9,971,뉴스를 보는 눈,풀빛,2026-05-20 09:05:30
10,1873,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2026-05-20 09:05:30


- UPDATE 를 합니다. WHERE 조건에 해당하는 행들을 바꿉니다.

In [9]:
%%SQL

UPDATE book_info SET publisher = '영국출판사' WHERE book_id = 1

'변경된 행의 수: 1'

In [10]:
%%SQL

SELECT * FROM book_info

,book_id,title,publisher,reg_date
1,1,NULL,영국출판사,2025-05-06 00:00:00
2,2459,수상한 과학,풀빛,2026-05-20 09:05:30
3,14,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2026-05-20 09:05:30
4,1384,2024 SIMPLE 의료법규,군자출판사,2026-05-20 09:05:30
5,704,임신부·모유수유부의 안전한 약물 사용,군자출판사,2026-05-20 09:05:30
6,1706,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2026-05-20 09:05:30
7,1065,상실의 시대,문학사상,2026-05-20 09:05:30
8,1544,"궁금해요, 세종 대왕",풀빛,2026-05-20 09:05:30
9,971,뉴스를 보는 눈,풀빛,2026-05-20 09:05:30
10,1873,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2026-05-20 09:05:30


In [11]:
%%SQL

UPDATE book_info SET reg_date = reg_date - 1

'변경된 행의 수: 12'

In [12]:
%%SQL

SELECT * FROM book_info

,book_id,title,publisher,reg_date
1,1,NULL,영국출판사,2025-05-05 00:00:00
2,2459,수상한 과학,풀빛,2026-05-19 09:05:30
3,14,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2026-05-19 09:05:30
4,1384,2024 SIMPLE 의료법규,군자출판사,2026-05-19 09:05:30
5,704,임신부·모유수유부의 안전한 약물 사용,군자출판사,2026-05-19 09:05:30
6,1706,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2026-05-19 09:05:30
7,1065,상실의 시대,문학사상,2026-05-19 09:05:30
8,1544,"궁금해요, 세종 대왕",풀빛,2026-05-19 09:05:30
9,971,뉴스를 보는 눈,풀빛,2026-05-19 09:05:30
10,1873,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2026-05-19 09:05:30


- DELETE FROM 테이블에서 조건해당하는 행을 삭제합니다.


In [13]:
%%SQL

DELETE FROM book_info WHERE book_id = (SELECT max(book_id) FROM book_info)

'변경된 행의 수: 1'

In [14]:
%%SQL

SELECT * FROM book_info

,book_id,title,publisher,reg_date
1,1,NULL,영국출판사,2025-05-05 00:00:00
2,2459,수상한 과학,풀빛,2026-05-19 09:05:30
3,14,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2026-05-19 09:05:30
4,1384,2024 SIMPLE 의료법규,군자출판사,2026-05-19 09:05:30
5,704,임신부·모유수유부의 안전한 약물 사용,군자출판사,2026-05-19 09:05:30
6,1706,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2026-05-19 09:05:30
7,1065,상실의 시대,문학사상,2026-05-19 09:05:30
8,1544,"궁금해요, 세종 대왕",풀빛,2026-05-19 09:05:30
9,971,뉴스를 보는 눈,풀빛,2026-05-19 09:05:30
10,1873,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2026-05-19 09:05:30


In [15]:
%%SQL

SELECT * FROM book1

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
2,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
3,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
4,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
5,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관",NULL,898,책읽는고양이,황윤,색다른 스토리텔링으로 만나는 국립중앙박물관그 주인공은 금동반가사유상이 책은 국립중앙...,2022-07-01,2022-07-01 10:34:15
7,14,차근차근 배우는 우리 아이 일상 말하기,스크립트 중재 기반,7,군자출판사,홍경훈|윤미선|이수향|유지원|김하은|정보 더 보기/감추기|홍경훈|윤미선|이수향|유지...,언어치료 전문가가 알려주는 우리 아이의 일상생활 말하기 연습!『차근차근 배우는 우리...,2023-02-01,2023-02-01 11:10:01
8,1384,2024 SIMPLE 의료법규,NULL,654,군자출판사,길벗,“법의 무지는 용서되지 않는다(Ignorantia juris non excusat)...,2023-05-01,2023-05-01 12:38:16
9,704,임신부·모유수유부의 안전한 약물 사용,모체뿐만 아니라 아이의 건강까지 고려한 안전한 약물 정보 가이드,324,군자출판사,한국마더세이프,『임신부·모유수유부의 안전한 약물 사용』은 고령 가임 여성의 증가와 만성질환으로 약...,2024-11-01,2024-11-01 13:33:39
10,3327,중학 국어 기초 완성,NULL,1579,꿈을담는틀,EBS,꼭 알아야 할 중학 국어 지식 총정리!중학생이 꼭 알아야 할 필수 개념만 모아 한 ...,2024-10-01,2024-10-01 12:06:54


In [16]:
%%SQL 

SELECT book_id, book_title, publisher, sysdate 
FROM book1
WHERE publisher like '%사'

,book_id,book_title,publisher,sysdate
1,14,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2026-05-20 09:05:45
2,1384,2024 SIMPLE 의료법규,군자출판사,2026-05-20 09:05:45
3,704,임신부·모유수유부의 안전한 약물 사용,군자출판사,2026-05-20 09:05:45


- SELECT 쿼리에서 반환된 행들을 입력합니다.

In [17]:
%%SQL
SELECT * FROM book_info

,book_id,title,publisher,reg_date
1,1,NULL,영국출판사,2025-05-05 00:00:00
2,2459,수상한 과학,풀빛,2026-05-19 09:05:30
3,14,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2026-05-19 09:05:30
4,1384,2024 SIMPLE 의료법규,군자출판사,2026-05-19 09:05:30
5,704,임신부·모유수유부의 안전한 약물 사용,군자출판사,2026-05-19 09:05:30
6,1706,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2026-05-19 09:05:30
7,1065,상실의 시대,문학사상,2026-05-19 09:05:30
8,1544,"궁금해요, 세종 대왕",풀빛,2026-05-19 09:05:30
9,971,뉴스를 보는 눈,풀빛,2026-05-19 09:05:30
10,1873,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2026-05-19 09:05:30


In [18]:
%%SQL

INSERT INTO book_info(book_id, title, publisher, reg_date)
SELECT book_id, book_title, publisher, sysdate 
FROM book1
WHERE publisher like '%사'

(cx_Oracle.IntegrityError) ORA-00001: unique constraint (LIB.PK_BOOK_INFO) violated
[SQL: INSERT INTO book_info(book_id, title, publisher, reg_date)
SELECT book_id, book_title, publisher, sysdate 
FROM book1
WHERE publisher like '%사']
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [19]:
%%SQL

SELECT * FROM book_info

,book_id,title,publisher,reg_date
1,1,NULL,영국출판사,2025-05-05 00:00:00
2,2459,수상한 과학,풀빛,2026-05-19 09:05:30
3,14,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2026-05-19 09:05:30
4,1384,2024 SIMPLE 의료법규,군자출판사,2026-05-19 09:05:30
5,704,임신부·모유수유부의 안전한 약물 사용,군자출판사,2026-05-19 09:05:30
6,1706,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2026-05-19 09:05:30
7,1065,상실의 시대,문학사상,2026-05-19 09:05:30
8,1544,"궁금해요, 세종 대왕",풀빛,2026-05-19 09:05:30
9,971,뉴스를 보는 눈,풀빛,2026-05-19 09:05:30
10,1873,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2026-05-19 09:05:30


In [20]:
%%SQL

UPDATE book_info SET title = '', publisher = publisher || '~~~' WHERE length(publisher) = 5

'변경된 행의 수: 5'

In [21]:
%%SQL

SELECT * FROM book_info

,book_id,title,publisher,reg_date
1,1,NULL,영국출판사~~~,2025-05-05 00:00:00
2,2459,수상한 과학,풀빛,2026-05-19 09:05:30
3,14,NULL,군자출판사~~~,2026-05-19 09:05:30
4,1384,NULL,군자출판사~~~,2026-05-19 09:05:30
5,704,NULL,군자출판사~~~,2026-05-19 09:05:30
6,1706,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2026-05-19 09:05:30
7,1065,상실의 시대,문학사상,2026-05-19 09:05:30
8,1544,"궁금해요, 세종 대왕",풀빛,2026-05-19 09:05:30
9,971,뉴스를 보는 눈,풀빛,2026-05-19 09:05:30
10,1873,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2026-05-19 09:05:30


In [22]:
%%SQL

SELECT * FROM book1

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
2,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
3,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
4,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
5,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관",NULL,898,책읽는고양이,황윤,색다른 스토리텔링으로 만나는 국립중앙박물관그 주인공은 금동반가사유상이 책은 국립중앙...,2022-07-01,2022-07-01 10:34:15
7,14,차근차근 배우는 우리 아이 일상 말하기,스크립트 중재 기반,7,군자출판사,홍경훈|윤미선|이수향|유지원|김하은|정보 더 보기/감추기|홍경훈|윤미선|이수향|유지...,언어치료 전문가가 알려주는 우리 아이의 일상생활 말하기 연습!『차근차근 배우는 우리...,2023-02-01,2023-02-01 11:10:01
8,1384,2024 SIMPLE 의료법규,NULL,654,군자출판사,길벗,“법의 무지는 용서되지 않는다(Ignorantia juris non excusat)...,2023-05-01,2023-05-01 12:38:16
9,704,임신부·모유수유부의 안전한 약물 사용,모체뿐만 아니라 아이의 건강까지 고려한 안전한 약물 정보 가이드,324,군자출판사,한국마더세이프,『임신부·모유수유부의 안전한 약물 사용』은 고령 가임 여성의 증가와 만성질환으로 약...,2024-11-01,2024-11-01 13:33:39
10,3327,중학 국어 기초 완성,NULL,1579,꿈을담는틀,EBS,꼭 알아야 할 중학 국어 지식 총정리!중학생이 꼭 알아야 할 필수 개념만 모아 한 ...,2024-10-01,2024-10-01 12:06:54


- MERGE INTO  구분을 사용해봅니다.

In [23]:
%%SQL
SELECT book_id, book_title, publisher, pub_date FROM book1

,book_id,book_title,publisher,pub_date
1,2459,수상한 과학,풀빛,2004-01-01
2,1706,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2004-07-01
3,1065,상실의 시대,문학사상,2000-10-01
4,1544,"궁금해요, 세종 대왕",풀빛,2019-07-01
5,971,뉴스를 보는 눈,풀빛,2019-10-01
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2022-07-01
7,14,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2023-02-01
8,1384,2024 SIMPLE 의료법규,군자출판사,2023-05-01
9,704,임신부·모유수유부의 안전한 약물 사용,군자출판사,2024-11-01
10,3327,중학 국어 기초 완성,꿈을담는틀,2024-10-01


In [24]:
%%SQL

MERGE INTO book_info a 
USING (SELECT book_id, book_title, publisher, pub_date FROM book1) b ON (a.book_id = b.book_id)
WHEN MATCHED THEN
UPDATE SET 
    title = b.book_title,  publisher = b.publisher
WHEN NOT MATCHED THEN
INSERT (book_id, title, publisher, reg_date)
VALUES (b.book_id, b.book_title, b.publisher, sysdate)

'변경된 행의 수: 10'

In [25]:
%%SQL

SELECT * FROM book_info

,book_id,title,publisher,reg_date
1,1,NULL,영국출판사~~~,2025-05-05 00:00:00
2,2459,수상한 과학,풀빛,2026-05-19 09:05:30
3,14,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2026-05-19 09:05:30
4,1384,2024 SIMPLE 의료법규,군자출판사,2026-05-19 09:05:30
5,704,임신부·모유수유부의 안전한 약물 사용,군자출판사,2026-05-19 09:05:30
6,1706,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2026-05-19 09:05:30
7,1065,상실의 시대,문학사상,2026-05-19 09:05:30
8,1544,"궁금해요, 세종 대왕",풀빛,2026-05-19 09:05:30
9,971,뉴스를 보는 눈,풀빛,2026-05-19 09:05:30
10,1873,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2026-05-19 09:05:30


# TCL(Transaction Control Language)

- 트랜젝션: 분리 될 수 없는 하나 이상의 데이터베이스 조작

## 트랜젝션의 특성

|명령|설명|
|-----|----|
|원자성(atomicity)|트랜잭션의 작업들은 모두 실행되거나 <br/>모두 실행되지 않아야 한다.|
|일관성(consistency)|트랜잭션 전의 내용에 이상이 없으면, <br/>트랜잭션 실행 후에도 이상이 없어야 한다.|
|고립성(isolation)|트랜잭션 실행 중에 다른 작업에 영향을 <br/>받아 결과에 이상이 생겨서는 안 된다.|
|지속성(durability)|트랜잭션 수행이 성공했다면, <br/> 갱신된 결과는 영구적이어야 한다.|

## 트랜젝션의 명령

|명령|설명|
|-----|----|
|COMMIT|변경 사항을 반영<br/>변경 이전 데이터 기억 해제<br/>모든 사용자가 변경 사항 조회 가능<br/>관련 행에 대한 잠금 해제. 타 사용자 조작가능|
|ROLLBACK<br/>\[포인트명\]|변경 사항 취소<br/>관련 행에 대한 잠금 해제. 타 사용자 조작가능<br/>포인트명을 지정하면, SAVEPOINT가 지정된 이후의 변경 사항 취소<br/>포인트명 미지정시, 모두 취소|
|SAVEPOINT<br/>포인트명|트랜잭션 내에서 ROLLBACK 시 복귀 지점 설정|


**[예제 2]** 트랜젝션 기능을 확인합니다.

- TRUNCATE 테이블은 Transaction에 포함되지 않고, 즉각 삭제 작업이 반영됩니다.

In [26]:
%%SQL

DELETE  FROM book_info;
-- ROLLBACK;

(cx_Oracle.DatabaseError) ORA-00900: invalid SQL statement
[SQL: -- ROLLBACK]
(Background on this error at: https://sqlalche.me/e/20/4xp6)


In [27]:
%%SQL

SELECT * FROM book_info

,book_id,title,publisher,reg_date
1,1,NULL,영국출판사~~~,2025-05-05 00:00:00
2,2459,수상한 과학,풀빛,2026-05-19 09:05:30
3,14,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2026-05-19 09:05:30
4,1384,2024 SIMPLE 의료법규,군자출판사,2026-05-19 09:05:30
5,704,임신부·모유수유부의 안전한 약물 사용,군자출판사,2026-05-19 09:05:30
6,1706,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2026-05-19 09:05:30
7,1065,상실의 시대,문학사상,2026-05-19 09:05:30
8,1544,"궁금해요, 세종 대왕",풀빛,2026-05-19 09:05:30
9,971,뉴스를 보는 눈,풀빛,2026-05-19 09:05:30
10,1873,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2026-05-19 09:05:30


In [28]:
%%SQL

INSERT INTO book_info(book_id, title, publisher, reg_date)
VALUES (1, '해리포터', '외국출판사', to_date('2025-05-07', 'yyyy-mm-dd'));

INSERT INTO book_info
VALUES (2, '반지의 제왕', '보석출판사', to_date('2025_06-01', 'yyyy-mm-dd'));

SAVEPOINT sp1;

INSERT INTO book_info (book_id, title, publisher, reg_date)
VALUES (3, '명탐정 코난', '일본출판사', SYSDATE);

ROLLBACK TO sp1;

COMMIT;

(cx_Oracle.IntegrityError) ORA-00001: unique constraint (LIB.PK_BOOK_INFO) violated
[SQL: INSERT INTO book_info(book_id, title, publisher, reg_date)
VALUES (1, '해리포터', '외국출판사', to_date('2025-05-07', 'yyyy-mm-dd'))]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [29]:
%%SQL

SELECT * FROM book_info

,book_id,title,publisher,reg_date
1,1,NULL,영국출판사~~~,2025-05-05 00:00:00
2,2459,수상한 과학,풀빛,2026-05-19 09:05:30
3,14,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2026-05-19 09:05:30
4,1384,2024 SIMPLE 의료법규,군자출판사,2026-05-19 09:05:30
5,704,임신부·모유수유부의 안전한 약물 사용,군자출판사,2026-05-19 09:05:30
6,1706,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2026-05-19 09:05:30
7,1065,상실의 시대,문학사상,2026-05-19 09:05:30
8,1544,"궁금해요, 세종 대왕",풀빛,2026-05-19 09:05:30
9,971,뉴스를 보는 눈,풀빛,2026-05-19 09:05:30
10,1873,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2026-05-19 09:05:30


- 비정상 종료시 롤백됩니다.

- 접속해제시 커밋됩니다.

In [30]:
%%SQL

INSERT INTO book_info(book_id, title, publisher, reg_date)
VALUES (4, '해리포터', '외국출판사', to_date('2025-05-07', 'yyyy-mm-dd'));

INSERT INTO book_info
VALUES (5, '반지의 제왕', '보석출판사', to_date('2025_06-01', 'yyyy-mm-dd'));

SAVEPOINT sp1;

INSERT INTO book_info (book_id, title, publisher, reg_date)
VALUES (4, '명탐정 코난', '일본출판사', SYSDATE);

ROLLBACK TO sp1;

COMMIT;

(cx_Oracle.IntegrityError) ORA-00001: unique constraint (LIB.PK_BOOK_INFO) violated
[SQL: INSERT INTO book_info (book_id, title, publisher, reg_date)
VALUES (4, '명탐정 코난', '일본출판사', SYSDATE)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [31]:
%%SQL

SELECT * FROM book_info

,book_id,title,publisher,reg_date
1,1,NULL,영국출판사~~~,2025-05-05 00:00:00
2,2459,수상한 과학,풀빛,2026-05-19 09:05:30
3,14,차근차근 배우는 우리 아이 일상 말하기,군자출판사,2026-05-19 09:05:30
4,1384,2024 SIMPLE 의료법규,군자출판사,2026-05-19 09:05:30
5,704,임신부·모유수유부의 안전한 약물 사용,군자출판사,2026-05-19 09:05:30
6,1706,큰글씨 나는 이렇게 나이들고 싶다,책읽는고양이,2026-05-19 09:05:30
7,1065,상실의 시대,문학사상,2026-05-19 09:05:30
8,1544,"궁금해요, 세종 대왕",풀빛,2026-05-19 09:05:30
9,971,뉴스를 보는 눈,풀빛,2026-05-19 09:05:30
10,1873,"일상이 고고학, 나 혼자 국립중앙박물관",책읽는고양이,2026-05-19 09:05:30


# DDL(Data Definition Language)

## CREATE TABLE

```SQL
CREATE TABLE 테이블명(
   컬럼명1   데이터유형 [기본값1] [NOT NULL]
   ,컬럼명2  데이터유형 [기본값2] [NOT NULL]
   ,…
   [, CONSTRAINT 제약명1 제약 조건1,  …]
);
```

**제약조건**

|종류|설명|
|----|-----|
|PRIMARY KEY|식별자로 사용하는 기본키 지정<br/>테이블 당 하나만 정의 가능<br/>UNIQUE와 NOT NULL 제약을 포함|
|UNIQUE|고유값 제약<br/>NULL은 제약 대상에서 빠진다|
|NOT NULL|NULL 값 미허용|
|CHECK|입력할 수 있는 값에 대한 제약|
|FOREIGN KEY|외래키 지정|


**명명 규칙**

<table>
  <tr>
    <th >테이블</th><th>컬럼</th>
  </tr>
  <tr>
    <td>객체 의미를 나타낼 수 있는 이름<br/>가능한 단수<br/>데이터베이스에서 중복불가</td>
    <td>테이블 내에서 중복 불가<br/>일관성 있게 사용하는 것을 권장</td>
  </tr>
  <tr>
    <td colspan="2">반드시 문자로 시작<br/>예약어 사용불가<br/>허용 글자: A-Z, a-z, 0-9, %, #<br/>대소문자를 구분하지 않지만, 내부적으로 대문자로 만들어짐</td>
  </tr>
</table>

```SQL
CREATE TABLE 테이블명 AS SELECT 구문
```


**[예제 3]** DDL의 기능을 확인해봅니다.

- book_stock 테이블을 생성합니다.

 CHECK 는 도메인의 조건을 나타냅니다.

In [32]:
%%SQL

CREATE TABLE book_stock (
    book_id NUMBER(12) NOT NULL,
    stock_seq NUMBER(10) CHECK(stock_seq > 0),
    book_status NUMBER(8) CHECK (book_status in (1, 2, 3, 4)),
    book_location VARCHAR(100) UNIQUE, 
    reg_date DATE,
    CONSTRAINT PK_BOOK_STOCK PRIMARY KEY (book_id, stock_seq),
    CONSTRAINT FK_BOOK_STOCK FOREIGN KEY (book_id) REFERENCES book(book_id) ON DELETE CASCADE
);

'변경된 행의 수: 0'

In [33]:
%%SQL

SELECT * FROM book_stock

,book_id,stock_seq,book_status,book_location,reg_date


In [34]:
%%SQL

SELECT * 
FROM book
FETCH FIRST 5 ROWS ONLY

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2164,황소 아저씨,NULL,1033,길벗어린이,권정생|정승각,추운 겨울밤에 황소 아저씨와 새앙쥐 남매들이 나누는 가슴 따뜻한 이야기. 엄마가 돌...,2001-01-01,2001-01-01 10:22:03
2,2830,명 1,역학의 맥,1385,갑을당,김상연,음양 오행을 바탕으로 전개되는 각종 이론과 명식작성법 및 해석방법 등을 오나전 초보...,2001-02-01,2001-02-01 09:41:49
3,2377,이승만 다시 보기,우리가 버린 건국의 아버지,1150,기파랑,인보길,이승만 다시 바라보기최근 이승만연구소를 발족시킨 인터넷신문 뉴데일리가 연구총서 제1...,2001-03-01,2001-03-01 16:53:22
4,2158,뭐하니?,NULL,1030,길벗어린이,유문조|최민오,'까꿍놀이'를 주제로 한 아기 그림책입니다. 책을 펼치면 구부정하게 등을 구부리고 ...,2001-05-01,2001-05-01 15:29:39
5,2849,돼지가 철학에 빠진 날,NULL,1394,김영사,스티븐 로|오숙은,고등학교 국민윤리 시간에 끝없이 나열되는 철학자들의 이름과 요약정리된 이론에 진절머...,2001-07-01,2001-07-01 09:27:44


- 레코드를 하나 넣어 봅니다.

In [35]:
%%SQL

INSERT INTO book_stock(book_id, stock_seq, book_status, book_location, reg_date)
VALUES (2924, 0, 1, '1 도서관', sysdate)

(cx_Oracle.IntegrityError) ORA-02290: check constraint (LIB.SYS_C008371) violated
[SQL: INSERT INTO book_stock(book_id, stock_seq, book_status, book_location, reg_date)
VALUES (2924, 0, 1, '1 도서관', sysdate)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


stock_seq > 0 인 도메인 조건에 어긋나 에러가 납니다.

In [36]:
%%SQL

INSERT INTO book_stock(book_id, stock_seq, book_status, book_location, reg_date)
VALUES (2924, 1, 1, '1 도서관', sysdate)

'변경된 행의 수: 1'

In [37]:
%%SQL

SELECT max(stock_seq) + 1 FROM book_stock WHERE book_id = 2924

MAX(STOCK_SEQ)+1    2
Name: 1, dtype: int64

- book_location에는 UNIQUE 제약이 있습니다. 위와 동일한 book_location을 넣어 봅니다.

In [38]:
%%SQL

INSERT INTO book_stock(book_id, stock_seq, book_status, book_location, reg_date)
VALUES
    (2924, (SELECT max(stock_seq) + 1 FROM book_stock WHERE book_id = 2924), 1, '1 도서관', sysdate)

(cx_Oracle.IntegrityError) ORA-00001: unique constraint (LIB.SYS_C008374) violated
[SQL: INSERT INTO book_stock(book_id, stock_seq, book_status, book_location, reg_date)
VALUES
    (2924, (SELECT max(stock_seq) + 1 FROM book_stock WHERE book_id = 2924), 1, '1 도서관', sysdate)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


- UNIQUE 제약이 어긋나 에러가 납니다.

In [39]:
%%SQL

INSERT INTO book_stock(book_id, stock_seq, book_status, book_location, reg_date)
VALUES
    (2924, (SELECT max(stock_seq) + 1 FROM book_stock WHERE book_id = 2924), 1, '2 도서관', sysdate)

'변경된 행의 수: 1'

In [40]:
%%SQL

INSERT INTO book_stock(book_id, stock_seq, book_status, book_location, reg_date)
VALUES
    (2231, (SELECT nvl(max(stock_seq), 0) + 1 FROM book_stock WHERE book_id = 2231), 1, '3 도서관', sysdate)

'변경된 행의 수: 1'

In [41]:
%%SQL

SELECT a.*, b.book_title, b.publisher
FROM book_stock a LEFT OUTER JOIN book b ON a.book_id =  b.book_id

,book_id,stock_seq,book_status,book_location,reg_date,book_title,publisher
1,2924,1,1,1 도서관,2026-05-20 09:05:45,결정적 사건으로 배우는 암호학,골든래빗
2,2924,2,1,2 도서관,2026-05-20 09:05:45,결정적 사건으로 배우는 암호학,골든래빗
3,2231,1,1,3 도서관,2026-05-20 09:05:45,내 작은 출판사를 소개합니다,세나북스


## ALTER TABLE

|명령|설명|
|-----|----|
|ADD |ALTER TABLE 테이블명<br/>ADD (추가할 컬럼명1 데이터유형 [기본 값] [NOT NULL]<br/>[, 추가할 컬럼명1 데이터유형 [기본 값] [NOT NULL]<br/>, …]);|
|DROP COLUMN|ALTER TABLE 테이블명 DROP (삭제할 컬럼1[, 삭제할 컬럼2, …]);<br/>MODIFY COLUMN	ALTER TABLE 테이블명<br/>MODIFY (컬럼명1 테이블유형1 [기본 값1] [NOT NULL]<br/>[, 컬럼명2, 테이블유형2 [기본 값1] [NOT NULL]<br/>, …]);|
|RENAME COLUMN|ALTER TABLE 테이블명 RENAME COLUMN 기존 컬럼명 TO  새로운 컬럼명|
|DROP CONSTRAINT|ALTER TABLE 테이블명 DROP CONSTRAINT 제약조건명;|
|ADD CONSTRAINT|ALTER TABLE 테이블명 ADD CONSTRAINT 제약조건명 제약조건;	|



**[예제 4]** ALTER TABLE의 기능을 알아 봅니다.

In [42]:
%%SQL

SELECT * FROM book_stock

,book_id,stock_seq,book_status,book_location,reg_date
1,2924,1,1,1 도서관,2026-05-20 09:05:45
2,2924,2,1,2 도서관,2026-05-20 09:05:45
3,2231,1,1,3 도서관,2026-05-20 09:05:45


- book_manager 컬럼을 추가합니다.

In [43]:
%%SQL

ALTER TABLE book_stock
ADD book_manager VARCHAR2(100)

'변경된 행의 수: 0'

In [44]:
%%SQL

SELECT * FROM book_stock

,book_id,stock_seq,book_status,book_location,reg_date,book_manager
1,2924,1,1,1 도서관,2026-05-20 09:05:45,NULL
2,2924,2,1,2 도서관,2026-05-20 09:05:45,NULL
3,2231,1,1,3 도서관,2026-05-20 09:05:45,NULL


- book_manager 컬럼을 book_man으로 바꿉니다.

In [45]:
%%SQL

ALTER TABLE book_stock
RENAME COLUMN book_manager TO book_man

'변경된 행의 수: 0'

In [46]:
%%SQL 

SELECT * FROM book_stock

,book_id,stock_seq,book_status,book_location,reg_date,book_man
1,2924,1,1,1 도서관,2026-05-20 09:05:45,NULL
2,2924,2,1,2 도서관,2026-05-20 09:05:45,NULL
3,2231,1,1,3 도서관,2026-05-20 09:05:45,NULL


- book_man 컬럼에 제약을 추가합니다.

In [47]:
%%SQL

ALTER TABLE book_stock
ADD CONSTRAINT book_stock_book_man_unique UNIQUE(book_man)

'변경된 행의 수: 0'

In [48]:
%%SQL

UPDATE book_stock SET book_man = '강선구';

(cx_Oracle.IntegrityError) ORA-00001: unique constraint (LIB.BOOK_STOCK_BOOK_MAN_UNIQUE) violated
[SQL: UPDATE book_stock SET book_man = '강선구']
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [49]:
%%SQL

UPDATE book_stock SET book_man = '강선구' || to_char(ROWNUM)

'변경된 행의 수: 3'

In [50]:
%%SQL

SELECT * FROM book_stock

,book_id,stock_seq,book_status,book_location,reg_date,book_man
1,2924,1,1,1 도서관,2026-05-20 09:05:45,강선구1
2,2924,2,1,2 도서관,2026-05-20 09:05:45,강선구2
3,2231,1,1,3 도서관,2026-05-20 09:05:45,강선구3


- 제약을 제거합니다.

In [51]:
%%SQL 

ALTER TABLE book_stock
DROP CONSTRAINT book_stock_book_man_unique

'변경된 행의 수: 0'

In [52]:
%%SQL

UPDATE book_stock SET book_man = '강선구' 

'변경된 행의 수: 3'

In [53]:
%%SQL

SELECT * FROM book_stock

,book_id,stock_seq,book_status,book_location,reg_date,book_man
1,2924,1,1,1 도서관,2026-05-20 09:05:45,강선구
2,2924,2,1,2 도서관,2026-05-20 09:05:45,강선구
3,2231,1,1,3 도서관,2026-05-20 09:05:45,강선구


- 컬럼을 제거합니다.

In [54]:
%%SQL

ALTER TABLE book_stock
DROP COLUMN book_man

'변경된 행의 수: 0'

In [55]:
%%SQL

SELECT * FROM book_stock

,book_id,stock_seq,book_status,book_location,reg_date
1,2924,1,1,1 도서관,2026-05-20 09:05:45
2,2924,2,1,2 도서관,2026-05-20 09:05:45
3,2231,1,1,3 도서관,2026-05-20 09:05:45


## 기타 DDL

|명령|설명|
|----|----|
|ALTER TABLE ~ RENAME TO ~|ALTER TABLE 기존 테이블명 RENAME TO 새로운 테이블명;<br/>sp_rename ‘기존 테이블명’, ‘새로운 테이블명‘;|
|DROP TABLE|DROP TABLE 테이블명 [CASCADE CONSTRAINT];|
|TRUNCATE TABLE|TRUNCATE TABLE 테이블명; <br/> 테이블은 사라지지 않고, 테이블 내용이 초기화됩니다. <br/>Transaction에 영향을 받지 않습니다.|


**[예제 5]** ALTER TABLE ~ RENAME TO, DROP TABLE, TRUNCATE TABLE을 알아봅니다.

In [56]:
%%SQL

ALTER TABLE book_stock RENAME TO book_stock_info

'변경된 행의 수: 0'

In [57]:
%%SQL

SELECT * FROM book_stock_info

,book_id,stock_seq,book_status,book_location,reg_date
1,2924,1,1,1 도서관,2026-05-20 09:05:45
2,2924,2,1,2 도서관,2026-05-20 09:05:45
3,2231,1,1,3 도서관,2026-05-20 09:05:45


In [58]:
%%SQL

TRUNCATE TABLE book_stock_info

'변경된 행의 수: 0'

In [59]:
%%SQL

DROP TABLE book_stock_info

'변경된 행의 수: 0'

# DCL

|기능|명령|
|----|----|
|권한 부여|GRANT 권한 TO {사용자ID|ROLE};|
|권한 회수|REVOKE 권한 FROM {사용자ID|ROLE};|
|역할 Role 생성|CREATE ROLE 역할명;|
|역할(Role) 부여|GRANT 역할명 TO 사용자ID;|
|역할(Role) 회수|REVOKE 역할명 FROM 사용자ID;|


- DBA 계정으로 접근을 하고 사용자를 추가 합니다.

In [60]:
DATABASE_URL = "oracle+cx_oracle://sys:multisqld@{}:1521/xe?mode=SYSDBA".format(domain_name)
engine = create_engine(DATABASE_URL)

In [61]:
%%SQL
ALTER SESSION SET CONTAINER = XEPDB1;

'변경된 행의 수: 0'

- test / test123 인 사용자를 추가합니다.

In [62]:
%%SQL

CREATE USER TEST IDENTIFIED BY test123;

(cx_Oracle.DatabaseError) ORA-01920: user name 'TEST' conflicts with another user or role name
[SQL: CREATE USER TEST IDENTIFIED BY test123]
(Background on this error at: https://sqlalche.me/e/20/4xp6)


In [63]:
%%SQL

ALTER USER TEST DEFAULT TABLESPACE LIB_SPACE

'변경된 행의 수: 0'

- 추가한 test / test123으로 접근합니다.

In [64]:
DATABASE_URL = "oracle+cx_oracle://test:test123@{}:1521/?service_name=XEPDB1".format(domain_name)
engine = create_engine(DATABASE_URL)

In [65]:
%%SQL

SELECT * FROM book1

ORA-00942: table or view does not exist None


SESSION을 만들 권한이 없어 logon 이 거부됩니다.

In [66]:
DATABASE_URL = "oracle+cx_oracle://sys:multisqld@{}:1521/xe?mode=SYSDBA".format(domain_name)
engine = create_engine(DATABASE_URL)

In [67]:
%%SQL
ALTER SESSION SET CONTAINER = XEPDB1;

'변경된 행의 수: 0'

- 접근 권한을 부여합니다.

In [68]:
%%SQL 

GRANT CONNECT TO TEST

'변경된 행의 수: 0'

- SESSION 생성 권한들 부여합니다.

In [69]:
%%SQL

GRANT CREATE SESSION TO TEST

'변경된 행의 수: 0'

- 테이블 공간에 대한 무한 접근 권한을 부여합니다.

In [70]:
%%SQL

GRANT UNLIMITED TABLESPACE TO TEST

'변경된 행의 수: 0'

In [71]:
DATABASE_URL = "oracle+cx_oracle://test:test123@{}:1521/?service_name=XEPDB1".format(domain_name)
engine = create_engine(DATABASE_URL)

In [72]:
%%SQL

SELECT * FROM tab

,tname,tabtype,clusterid


권한을 받으면 정상 접근이 됩니다.

In [73]:
DATABASE_URL = "oracle+cx_oracle://sys:multisqld@{}:1521/xe?mode=SYSDBA".format(domain_name)
engine = create_engine(DATABASE_URL)

In [74]:
%%SQL
ALTER SESSION SET CONTAINER = XEPDB1;

'변경된 행의 수: 0'

- TEST 유저에 lib.book1에 SELECT 권한을 부여합니다.

In [75]:
%%SQL

GRANT SELECT ON lib.book1 TO TEST;

'변경된 행의 수: 0'

In [76]:
DATABASE_URL = "oracle+cx_oracle://test:test123@{}:1521/?service_name=XEPDB1".format(domain_name)
engine = create_engine(DATABASE_URL)

- book1의 내용을 조회합니다.

In [77]:
%%SQL

SELECT * FROM lib.book1

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
2,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
3,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
4,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
5,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관",NULL,898,책읽는고양이,황윤,색다른 스토리텔링으로 만나는 국립중앙박물관그 주인공은 금동반가사유상이 책은 국립중앙...,2022-07-01,2022-07-01 10:34:15
7,14,차근차근 배우는 우리 아이 일상 말하기,스크립트 중재 기반,7,군자출판사,홍경훈|윤미선|이수향|유지원|김하은|정보 더 보기/감추기|홍경훈|윤미선|이수향|유지...,언어치료 전문가가 알려주는 우리 아이의 일상생활 말하기 연습!『차근차근 배우는 우리...,2023-02-01,2023-02-01 11:10:01
8,1384,2024 SIMPLE 의료법규,NULL,654,군자출판사,길벗,“법의 무지는 용서되지 않는다(Ignorantia juris non excusat)...,2023-05-01,2023-05-01 12:38:16
9,704,임신부·모유수유부의 안전한 약물 사용,모체뿐만 아니라 아이의 건강까지 고려한 안전한 약물 정보 가이드,324,군자출판사,한국마더세이프,『임신부·모유수유부의 안전한 약물 사용』은 고령 가임 여성의 증가와 만성질환으로 약...,2024-11-01,2024-11-01 13:33:39
10,3327,중학 국어 기초 완성,NULL,1579,꿈을담는틀,EBS,꼭 알아야 할 중학 국어 지식 총정리!중학생이 꼭 알아야 할 필수 개념만 모아 한 ...,2024-10-01,2024-10-01 12:06:54


In [78]:
DATABASE_URL = "oracle+cx_oracle://sys:multisqld@{}:1521/xe?mode=SYSDBA".format(domain_name)
engine = create_engine(DATABASE_URL)

In [79]:
%%SQL
ALTER SESSION SET CONTAINER = XEPDB1;

'변경된 행의 수: 0'

- TEST에게서 lib.book1에 대한 SELECT 권한을 회수합니다. 

In [80]:
%%SQL

REVOKE SELECT ON lib.book1 FROM TEST;

'변경된 행의 수: 0'

In [81]:
DATABASE_URL = "oracle+cx_oracle://test:test123@{}:1521/?service_name=XEPDB1".format(domain_name)
engine = create_engine(DATABASE_URL)

In [82]:
%%SQL

SELECT * FROM lib.book1

ORA-00942: table or view does not exist None


In [83]:
DATABASE_URL = "oracle+cx_oracle://sys:multisqld@{}:1521/xe?mode=SYSDBA".format(domain_name)
engine = create_engine(DATABASE_URL)

In [84]:
%%SQL
ALTER SESSION SET CONTAINER = XEPDB1;

'변경된 행의 수: 0'

In [85]:
%%SQL
SELECT s.sid, s.serial#, s.username, s.status, s.osuser, s.machine
FROM v$session s
WHERE s.username = 'TEST'

,sid,SERIAL#,username,status,osuser,machine
1,293,37246,TEST,INACTIVE,root,7cf52d4de7af
2,298,20794,TEST,INACTIVE,root,7cf52d4de7af


In [86]:
"""
%%SQL

ALTER SYSTEM KILL SESSION '282,60825' IMMEDIATE
"""

"\n%%SQL\n\nALTER SYSTEM KILL SESSION '282,60825' IMMEDIATE\n"

In [87]:
%%SQL

DROP USER test

(cx_Oracle.DatabaseError) ORA-01940: cannot drop a user that is currently connected
[SQL: DROP USER test]
(Background on this error at: https://sqlalche.me/e/20/4xp6)
